# Convolutional Neural Networks (CNNs)

Reach for this when you need: 
- Reference for spatial feature extraction layers.
- To understand weight sharing and translational invariance.
- Implementation of standard pooling and normalization layers.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Core CNN Layers

| Layer | Description | Industry Usage |
| :--- | :--- | :--- |
| `nn.Conv2d` | Sliding window of learnable weights | Feature extraction |
| `nn.MaxPool2d` | Downsampling by max value | Reducing spatial size, increasing RF |
| `nn.BatchNorm2d` | Internal covariate shift reduction | Speeds up convergence, stabilizes training |
| `nn.AdaptiveAvgPool2d` | Fixed-size downsampling | Ensuring fixed output size for any input size |

In [2]:
# nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)
conv = nn.Conv2d(3, 16, kernel_size=3, padding=1) 

# Formula for output size: Floor((W - K + 2P)/S + 1)
# If padding=kernel_size//2 and stride=1, output size == input size.

x = torch.randn(1, 3, 32, 32) # NCHW format
out = conv(x)
print(out.shape)

torch.Size([1, 16, 32, 32])


## 2. The Residual Connection (ResNet)

The most successful CNN architectural trick. Prevents vanishing gradients in deep networks.

✅ **Use when**: Any network deeper than 10–20 layers.
❌ **Don't use when**: Shallow networks where the identity mapping adds unnecessary compute.

In [3]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x # Store the identity
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual # The skip connection
        return F.relu(out)

### Common Pitfalls
- **Shape Mismatch**: Forgetting that Conv2d expects 4D tensors `(N, C, H, W)`. Use `x.unsqueeze(0)` if you have a single image.
- **Channel ordering**: Many CV libraries use `(H, W, C)`, but PyTorch expects `(C, H, W)`. Always transpose/permute incoming images.
- **BatchNorm usage**: Ensure `model.eval()` is called for inference, as BatchNorm behavior differs significantly during train vs. test.

### Key Takeaways
- `BatchNorm2d` should generally follow `Conv2d` but precede the activation function (e.g. `ReLU`).
- `AdaptiveAvgPool2d((1, 1))` is standard for transforming a feature map into a vector for a Linear classification head.
- Pooling layers have no learnable parameters, making them a cheap way to increase the Receptive Field (RF).